# Question 1 - Linear Models and Maximum Likelihood

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np                 # math and arrays
import pandas as pd                # reading the CSV
import matplotlib.pyplot as plt    # plots

In [ ]:
HW_DIR = "/content/drive/MyDrive/Computational Models - HW2"   # assignment folder in Drive
DATA_PATH = f"{HW_DIR}/data/HW2_q1_data.csv"                   # Q1 data file

df = pd.read_csv(DATA_PATH)        # load the table (columns: x1, y1)
x = df["x1"].to_numpy()            # stimulus
y = df["y1"].to_numpy()            # firing rate
N = len(y)                         # number of samples

print("N =", N)
df.head()

In [ ]:
X = np.column_stack([np.ones(N), x])   # N x 2; the column of ones carries the intercept theta0
X[:3]

## Task a

In [ ]:
# theta = (X^T X)^-1 X^T y   ->  solved as a linear system (more stable than inverting)
theta_ols = np.linalg.solve(X.T @ X, X.T @ y)
theta0, theta1 = theta_ols

print(f"theta0 (intercept) = {theta0:.4f}")
print(f"theta1 (slope)     = {theta1:.4f}")

## Task b

In [ ]:
learning_rate = 0.01      # step size (eta)
n_epochs = 10_000         # number of update steps

theta = np.zeros(2)                        # start from [0, 0]
losses = []                                # MSE after every epoch

for epoch in range(n_epochs):
    residuals = X @ theta - y              # prediction errors
    grad = (2 / N) * X.T @ residuals       # gradient of the MSE
    theta = theta - learning_rate * grad   # update step
    losses.append(np.mean((X @ theta - y) ** 2))

theta_gd = theta
print(f"learning rate = {learning_rate}, epochs = {n_epochs}")
print(f"GD:         theta0 = {theta_gd[0]:.4f}, theta1 = {theta_gd[1]:.4f}")
print(f"Analytical: theta0 = {theta_ols[0]:.4f}, theta1 = {theta_ols[1]:.4f}")
print(f"max |difference| = {np.max(np.abs(theta_gd - theta_ols)):.2e}")

## Task c

In [ ]:
y_hat = X @ theta_ols                      # model predictions

mse = np.mean((y - y_hat) ** 2)            # mean squared error
ss_res = np.sum((y - y_hat) ** 2)          # unexplained variance
ss_tot = np.sum((y - np.mean(y)) ** 2)     # total variance
r2 = 1 - ss_res / ss_tot                   # explained variance

print(f"MSE = {mse:.4f}")
print(f"R^2 = {r2:.4f}")

## Task d

In [ ]:
# LL = -N/2 * ln(2*pi*sigma^2) - SSE / (2*sigma^2)
sigma2 = mse
sse = np.sum((y - y_hat) ** 2)
ll = -N / 2 * np.log(2 * np.pi * sigma2) - sse / (2 * sigma2)

print(f"sigma^2 = MSE = {sigma2:.4f}")
print(f"SSE = {sse:.4f}")
print(f"LL = -{N}/2 * ln(2*pi*{sigma2:.4f}) - {sse:.4f}/(2*{sigma2:.4f})")
print(f"   = {-N / 2 * np.log(2 * np.pi * sigma2):.4f} - {sse / (2 * sigma2):.4f}")
print(f"   = {ll:.4f}")

## Task e

In [ ]:
x_line = np.linspace(x.min(), x.max(), 200)        # smooth x values for the line
y_line = theta0 + theta1 * x_line                  # fitted line

plt.figure(figsize=(7, 5))
plt.scatter(x, y, alpha=0.8, label="Data")
plt.plot(x_line, y_line, color="red", linewidth=2,
         label=f"Fit: y = {theta0:.3f} + {theta1:.3f}x")
plt.title("Q1 - Neuron firing rate vs. stimulus: linear fit")
plt.xlabel("Stimulus (x1)")
plt.ylabel("Firing rate (y1)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{HW_DIR}/q1_linear_fit.png", dpi=150)  # save into the assignment folder
plt.show()